# Environment and persistent paths
Run cells in order. Notebook outputs remain visible; durable artifacts are written to results/v2.


In [ ]:
from pathlib import Path
import os, sys, json, errno, time
# VS Code + Colab: load the synchronized source folder, without browser uploads.
DRIVE = Path("/content/drive/MyDrive")
try:
    from google.colab import drive
except ImportError:
    drive = None
def with_drive_retry(operation, attempts=3, reconnect=None):
    for attempt in range(attempts):
        try:
            return operation()
        except OSError as exc:
            disconnected = exc.errno in {errno.ENOTCONN, errno.EIO, errnoNotebook: Select Notebook Kernel
                                            .ESTALE} or "Transport endpoint" in str(exc)
            if not disconnected or (drive is None and reconnect is None) or attempt == attempts-1:
                raise
            print(f"Drive disconnected. Reconnecting and retrying ({attempt+1}/{attempts-1})...", flush=True)
            if reconnect is not None:
                reconnect()
            else:
                drive.mount("/content/drive", force_remount=True)
            time.sleep(1)

def ensure_drive():
    if drive is not None:
        if not DRIVE.is_dir():
            drive.mount("/content/drive", force_remount=False)
        # Probe the filesystem: an existing mount directory can still be disconnected.
        next(DRIVE.iterdir(), None)
with_drive_retry(ensure_drive)

def resolve_v2(bases):
    checked = []
    for base in bases:
        for candidate in (base, base/"implementation/v2", base/"v2"):
            checked.append(str(candidate))
            required = ["src/atlas_v2/__init__.py", "src/atlas_v2/data.py",
                        "src/atlas_v2/models.py", "src/atlas_v2/experiments.py", "requirements.txt"]
            if all((candidate/name).is_file() for name in required):
                return candidate.resolve()
    raise FileNotFoundError(
        "The v2 Python source is not visible to this kernel. "
        "Expected the synchronized implementation/v2 folder on mounted Drive. "
        "Set ATLAS_V2_ROOT to that folder if your Drive location differs. Checked: "
        + "; ".join(checked))

manual = os.environ.get("ATLAS_V2_ROOT")
bases = [Path(manual).expanduser()] if manual else []
bases += [DRIVE/"projects/implementation/ATLAS-IDNet/implementation/v2",
          DRIVE/"projects/ATLAS-IDNet", DRIVE/"ATLAS-IDNet",
          Path.cwd(), *Path.cwd().parents, Path("/content/ATLAS-IDNet")]
V2 = with_drive_retry(lambda: resolve_v2(bases))
sys.path.insert(0, str(V2/"src"))
for name in list(sys.modules):
    if name == "atlas_v2" or name.startswith("atlas_v2."):
        del sys.modules[name]
import atlas_v2
assert Path(atlas_v2.__file__).resolve().parent == (V2/"src/atlas_v2").resolve()
print("Code:", V2)
print("Source import OK. No file upload or model weights required.")


In [2]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(V2/"requirements.txt")])
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive")
import torch, pandas, sklearn
print("CPU runtime:", torch.__version__, "threads:", os.cpu_count())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CPU runtime: 2.11.0+cpu threads: 2


In [3]:
# Review these paths before running. Results go to a NEW v2 directory.
DRIVE = Path("/content/drive/MyDrive")
BASE = DRIVE/"projects/Dataset/Security" if DRIVE.exists() else Path("/Volumes/LocalDrive1/Dataset/Security")
RESULTS = DRIVE/"projects/implementation/ATLAS-IDNet/results/v2" if DRIVE.exists() else V2/"results"
RESULTS.mkdir(parents=True, exist_ok=True)
CONFIG = {
    "results": str(RESULTS), "seeds": [42, 43, 44, 45, 46], "epochs": 30,
    "datasets": {
        "CICIoT2023": {"csv_globs": [str(BASE/"CICIoT2023/**/*.csv")], "label": "label"},
        "NF-UQ-NIDS-v2": {"csv_globs": [str(BASE/"NF-UQ-NIDS-v2/**/*.csv")], "label": "Attack"}
    }
}
for spec in CONFIG["datasets"].values():
    spec.update(max_rows=100000, split="stratified", group_column=None, time_column=None,
        exclude_columns=["Label", "label", "Attack", "Attack_type", "Class", "Dataset", "dataset",
                         "IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Flow ID", "Src IP", "Dst IP", "Timestamp"])
CONFIG_PATH = RESULTS/"experiment_config.json"
if CONFIG_PATH.exists():
    CONFIG = json.loads(CONFIG_PATH.read_text())
    print("Reused existing config; edit explicitly to change paths/protocols.")
else:
    CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))
os.environ["ATLAS_V2_CONFIG"] = str(CONFIG_PATH)


{
  "results": "/content/drive/MyDrive/projects/implementation/ATLAS-IDNet/results/v2",
  "seeds": [
    42,
    43,
    44,
    45,
    46
  ],
  "epochs": 30,
  "datasets": {
    "CICIoT2023": {
      "csv_globs": [
        "/content/drive/MyDrive/projects/Dataset/Security/CICIoT2023/**/*.csv"
      ],
      "label": "label",
      "max_rows": 100000,
      "split": "stratified",
      "group_column": null,
      "time_column": null,
      "exclude_columns": [
        "Label",
        "label",
        "Attack",
        "Attack_type",
        "Class",
        "Dataset",
        "dataset",
        "IPV4_SRC_ADDR",
        "IPV4_DST_ADDR",
        "Flow ID",
        "Src IP",
        "Dst IP",
        "Timestamp"
      ]
    },
    "NF-UQ-NIDS-v2": {
      "csv_globs": [
        "/content/drive/MyDrive/projects/Dataset/Security/NF-UQ-NIDS-v2/**/*.csv"
      ],
      "label": "Attack",
      "max_rows": 100000,
      "split": "stratified",
      "group_column": null,
      "time_column":